# 📉 Ridge Regression: Optimization via Gradient Descent
> **Objective:** Comparing Scikit-Learn's Stochastic Gradient Descent (SGD) with a custom-built Batch Gradient Descent solver to implement L2 Regularization.

---
## 🧪 Concept Overview
In this notebook, we explore how models "learn" iteratively. While the Normal Equation gives an exact solution, **Gradient Descent** is the engine behind modern Deep Learning, allowing us to handle massive datasets efficiently.

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/arunjathari/bostonhousepricedata/Boston-house-price-data.csv


In [2]:
df = pd.read_csv("/kaggle/input/datasets/arunjathari/bostonhousepricedata/Boston-house-price-data.csv")

In [3]:
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [4]:
df.shape

(506, 14)

In [5]:
X = df.iloc[:,0:13]
y = df['MEDV']

In [6]:
X

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273.0,21.0,391.99,9.67
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273.0,21.0,396.90,9.08
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273.0,21.0,396.90,5.64
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273.0,21.0,393.45,6.48


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
scalar = StandardScaler()

In [10]:
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

## 🤖 Part 1: Scikit-Learn's SGDRegressor
We start by using the built-in `SGDRegressor`. We use `penalty='l2'` to invoke Ridge regularization and tune the `learning_rate` to ensure convergence without divergence.

In [11]:
reg = SGDRegressor(penalty='l2', max_iter=500, eta0=0.01, learning_rate='constant',alpha=0.001)

In [12]:
reg.fit(X_train_scaled, y_train)

y_pred = reg.predict(X_test_scaled)

print("R2 score: ",r2_score(y_test, y_pred))
print("Coef_ : ", reg.coef_)
print("Intercept_ : ", reg.intercept_)


R2 score:  0.631307949261285
Coef_ :  [-0.84306862  0.83470874  0.20348379  1.23201796 -2.0779234   3.92815528
 -0.02198961 -3.18974729  2.32272676 -1.45430858 -2.31086101  1.42825668
 -3.5248199 ]
Intercept_ :  [23.45103903]


In [13]:
from sklearn.linear_model import Ridge

rr = Ridge(alpha=0.001, max_iter=500, solver='sparse_cg')

In [14]:
rr.fit(X_train_scaled, y_train)

y_pred = rr.predict(X_test_scaled)

print("R2 score: ",r2_score(y_test, y_pred))
print("Coef_ : ", rr.coef_)
print("Intercept_ : ", rr.intercept_)


R2 score:  0.6687454670448303
Coef_ :  [-1.00165814  0.69624793  0.28092536  0.71864101 -2.02408036  3.1452956
 -0.1743467  -3.08021082  2.25044949 -1.76662145 -2.0380544   1.12942504
 -3.61254853]
Intercept_ :  22.796534653465343


## 🏗️ Part 2: Custom Ridge GD Implementation
Now, we translate the mathematical theory into a NumPy class.

### 📐 The Gradient Formula:
To update our weights ($W$) and bias ($b$), we calculate the mean gradient across all samples:
1. **MSE Gradient:** $\frac{1}{n} X^T(XW - y)$
2. **L2 Penalty:** $\frac{\alpha}{n} W$ (Note: We do not penalize the intercept/bias).

By dividing the gradient by $n$, we stabilize the updates, allowing for a more robust learning process.

In [15]:
class Myridge:
    def __init__(self, alpha, learning_rate, epochs):
        self.alpha = alpha
        self.epochs = epochs
        self.lr = learning_rate
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):
        # 1. Initialize theta (Weights + Intercept)
        theta = np.zeros(X_train.shape[1] + 1) # Start with zeros for stability
        
        # 2. Add bias column
        X_train_bias = np.insert(X_train, 0, 1, axis=1)
        
        n = len(X_train)
        
        for i in range(self.epochs):
            y_pred = np.dot(X_train_bias, theta)
            error = y_pred - y_train
            
            # Intercept ko regularize nahi karna hai
            penalty_theta = np.copy(theta)
            penalty_theta[0] = 0
            
            # Gradient calculation with MEAN (Dividing by n is the secret!)
            gradient = (np.dot(X_train_bias.T, error) + (self.alpha * penalty_theta)) / n
            
            # Update weights
            theta = theta - self.lr * gradient
            
        self.coef_ = theta[1:]
        self.intercept_ = theta[0]

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

In [16]:
reg = Myridge(epochs=500, alpha=0.01, learning_rate =0.01)

In [17]:
reg.fit(X_train_scaled, y_train)

y_pred = reg.predict(X_test_scaled)

print("R2 score: ",r2_score(y_test, y_pred))
print("Coef_ : ", reg.coef_)
print("Intercept_ : ", reg.intercept_)


R2 score:  0.6406531436549213
Coef_ :  [-0.76410241  0.2279284  -0.22619672  0.8045384  -1.07869962  3.49307904
 -0.17415642 -2.09419616  0.72817403 -0.44055231 -1.81939461  1.1308217
 -3.43798762]
Intercept_ :  22.64675040909894


In [18]:
# Epochs ko 500 se badhakar 2000-5000 karke dekhiye
reg = Myridge(epochs=5000, alpha=0.001, learning_rate=0.1)

In [19]:
reg.fit(X_train_scaled, y_train)

y_pred = reg.predict(X_test_scaled)

print("R2 score: ",r2_score(y_test, y_pred))
print("Coef_ : ", reg.coef_)
print("Intercept_ : ", reg.intercept_)

R2 score:  0.668759197071964
Coef_ :  [-1.00212511  0.69624951  0.27803804  0.71874229 -2.02228736  3.14524621
 -0.17604918 -3.08187016  2.25132562 -1.76693929 -2.03774142  1.12956593
 -3.61164557]
Intercept_ :  22.796534653465333


## 🏁 Final Comparison & Results
We have successfully matched the performance of industry-standard libraries with our scratch implementation.

| Model | $R^2$ Score | Intercept |
| :--- | :--- | :--- |
| **Scikit-Learn Ridge** | 0.6687 | 22.79 |
| **Custom Myridge (Batch GD)** | 0.6687 | 22.79 |

### 🚀 Key Takeaways:
1. **Mathematical Validation:** Our custom class produces identical coefficients to Scikit-Learn, verifying our derivative logic.
2. **Stability:** Feature scaling is mandatory for Gradient Descent to prevent "Exploding Gradients."
3. **Efficiency:** For this small dataset, 5000 epochs with a learning rate of 0.1 provided perfect convergence.